# Build Southern Lake Michigan NDBC Station Metadata

This notebook creates a reusable metadata table for NDBC buoys and fixed
platforms in southern Lake Michigan.

The NDBC active-station `met` flag only indicates whether a station has
reported meteorological data recently. It does **not** describe the variables
available in the station's historical record. Great Lakes buoys are often
seasonal, so a station can have useful wind, temperature, or radiation data
while its current `met` flag is `n`.

This workflow inspects archived files and derives four broad capability flags:

- `has_wind`
- `has_water_temperature`
- `has_air_temperature`
- `has_radiation`

The resulting CSV can be used by mapping and meteorological-analysis
notebooks.


## Workflow

1. Load the active NDBC station table.
2. Select stations in a southern Lake Michigan bounding box.
3. Apply the same manual exclusions used in the exploratory notebook.
4. Inspect several recent historical years with
   `ndbc_io.inspect_station_capabilities()`.
5. Audit the resulting table.
6. Export `data/se_lake_michigan_buoy_metadata.csv`.

The low-level archive readers and capability tests live in `ndbc_io.py`.
Regional selection, manual exclusions, and CSV export remain in this notebook.


In [1]:
from pathlib import Path

import importlib
import numpy as np
import pandas as pd

import ndbc_io as buoys

importlib.reload(buoys)


<module 'ndbc_io' from '/home/anderson/py/great_lakes/ndbc_io.py'>

In [2]:
OUTPUT_FILE = Path(
    "data/se_lake_michigan_buoy_metadata.csv"
)

CAPABILITY_YEARS = (
    2025,
    2024,
    2023,
    2022,
)

# Southern Lake Michigan station-selection bounds.
MIN_LATITUDE = 41.5
MAX_LATITUDE = 42.6
MIN_LONGITUDE = -88.0
MAX_LONGITUDE = -84.7


## Select Southern Lake Michigan stations

The active-station table is used for station identity, location, ownership,
and fixed-versus-floating designation. Station IDs are normalized to uppercase
for consistency with downstream programs.


In [3]:
active_stations = buoys.list_realtime_stations()

mask = (
    active_stations["lat"].between(
        MIN_LATITUDE,
        MAX_LATITUDE,
        inclusive="neither",
    )
    & active_stations["lon"].between(
        MIN_LONGITUDE,
        MAX_LONGITUDE,
        inclusive="neither",
    )
)

stations = active_stations.loc[mask].copy()

print(f"Stations inside bounding box: {len(stations)}")
display(stations)


Stations inside bounding box: 22


,id,lat,lon,elev,name,owner,pgm,type,met,currents,waterquality,dart
14,18ci3,41.730,-86.910,176.0,"Michigan City CG Station, IN",U.S.C.G. Marine Reporting Stations,IOOS Partners,fixed,n,n,n,n
15,20cm4,42.090,-86.490,177.0,"St. Joseph CG Station, MI",U.S.C.G. Marine Reporting Stations,IOOS Partners,fixed,n,n,n,n
266,45026,41.982,-86.619,176.0,"Cook Nuclear Plant Buoy, Stevensville, MI",Limno Tech,IOOS Partners,buoy,y,y,y,n
296,45168,42.397,-86.331,177.0,"South Haven Buoy, MI",Limno Tech,IOOS Partners,buoy,y,n,y,n
297,45170,41.755,-86.968,177.0,"Michigan City Buoy, IN",Illinois-Indiana Sea Grant and Purdue Civil En...,IOOS Partners,buoy,y,n,y,n
301,45174,42.135,-87.655,176.0,"Wilmette Buoy, IL",Illinois-Indiana Sea Grant,IOOS Partners,buoy,y,n,y,n
304,45177,41.894,-87.613,176.0,"Ohio St. Beach, Chicago, IL",Chicago Park District,IOOS Partners,buoy,n,n,n,n
309,45186,42.368,-87.795,176.0,"Waukegan Buoy, IL",University of Illinois at Urbana-Champaign,IOOS Partners,buoy,y,y,n,n
310,45187,42.491,-87.779,176.0,"Winthrop Harbor Buoy, IL",University of Illinois at Urbana-Champaign,IOOS Partners,buoy,y,y,n,n
317,45198,41.892,-87.563,176.0,Chicago Buoy,Illinois-Indiana Sea Grant and Purdue Civil En...,IOOS Partners,buoy,y,n,y,n


## Apply known station exclusions

These exclusions are retained from `buoy_exploration.ipynb`. They remove
stations known to have stopped collecting data well before the current study
period.

This is a project-specific editorial decision and therefore remains in the
notebook rather than in `ndbc_io.py`.


In [4]:
EXCLUDED_STATIONS = {
    "FSTI2": "Data collection ended in 2021",
    "45177": "Data collection ended in 2019",
    "JAKI2": "Data collection ended in 2022",
    "18CI3": "Data collection ended in 2015",
    "20CM4": "Data collection ended in 2015",
}

stations["id"] = (
    stations["id"]
    .astype(str)
    .str.upper()
)

stations = stations.loc[
    ~stations["id"].isin(EXCLUDED_STATIONS)
].copy()

stations = stations.reset_index(drop=True)

print(f"Stations retained: {len(stations)}")
display(stations)


Stations retained: 17


,id,lat,lon,elev,name,owner,pgm,type,met,currents,waterquality,dart
0,45026,41.982,-86.619,176.0,"Cook Nuclear Plant Buoy, Stevensville, MI",Limno Tech,IOOS Partners,buoy,y,y,y,n
1,45168,42.397,-86.331,177.0,"South Haven Buoy, MI",Limno Tech,IOOS Partners,buoy,y,n,y,n
2,45170,41.755,-86.968,177.0,"Michigan City Buoy, IN",Illinois-Indiana Sea Grant and Purdue Civil En...,IOOS Partners,buoy,y,n,y,n
3,45174,42.135,-87.655,176.0,"Wilmette Buoy, IL",Illinois-Indiana Sea Grant,IOOS Partners,buoy,y,n,y,n
4,45186,42.368,-87.795,176.0,"Waukegan Buoy, IL",University of Illinois at Urbana-Champaign,IOOS Partners,buoy,y,y,n,n
5,45187,42.491,-87.779,176.0,"Winthrop Harbor Buoy, IL",University of Illinois at Urbana-Champaign,IOOS Partners,buoy,y,y,n,n
6,45198,41.892,-87.563,176.0,Chicago Buoy,Illinois-Indiana Sea Grant and Purdue Civil En...,IOOS Partners,buoy,y,n,y,n
7,BHRI3,41.646,-87.147,180.0,"Burns Harbor, IN","NWS WFO Chicago, IL (LOT)",IOOS Partners,fixed,y,n,n,n
8,CHII2,41.916,-87.572,176.0,"Harrison-Dever Crib, Chicago, IL",GLERL,IOOS Partners,fixed,y,n,n,n
9,CMTI2,41.730,-87.538,178.5,"9087044 - Calumet Harbor, IL",NOS,NOS/CO-OPS,fixed,y,n,n,n


## Inspect historical measurement capabilities

Several years are inspected because archive availability varies and many
Great Lakes buoys are removed during winter. A capability is marked `True`
when at least one plausible archived observation is found in the requested
years.

A `False` value means only that no qualifying value was found in the archives
successfully read during this inspection. It is not proof that the station has
never measured the variable.


In [5]:
records = []

for station in stations.itertuples(index=False):
    station_id = str(station.id).upper()

    print(f"Checking {station_id} ...")

    capabilities = buoys.inspect_station_capabilities(
        station_id,
        years=CAPABILITY_YEARS,
    )

    records.append(
        {
            "station_id": station_id,
            "name": station.name,
            "latitude": station.lat,
            "longitude": station.lon,
            "elevation": getattr(
                station,
                "elev",
                np.nan,
            ),
            "owner": getattr(
                station,
                "owner",
                "",
            ),
            "station_type": str(
                getattr(station, "type", "")
            ).lower(),
            **capabilities,
        }
    )

buoy_metadata = pd.DataFrame.from_records(records)

BOOLEAN_COLUMNS = [
    "has_wind",
    "has_water_temperature",
    "has_air_temperature",
    "has_radiation",
]

buoy_metadata[BOOLEAN_COLUMNS] = (
    buoy_metadata[BOOLEAN_COLUMNS]
    .fillna(False)
    .astype(bool)
)

buoy_metadata = (
    buoy_metadata
    .sort_values("station_id")
    .reset_index(drop=True)
)

display(buoy_metadata)


Checking 45026 ...
Checking 45168 ...
Checking 45170 ...
Checking 45174 ...
Checking 45186 ...
Checking 45187 ...
Checking 45198 ...
Checking BHRI3 ...
Checking CHII2 ...
Checking CMTI2 ...
Checking CNII2 ...
Checking KNSW3 ...
Checking MCYI3 ...
Checking OKSI2 ...
Checking SJOM4 ...
Checking SVNM4 ...
Checking WHRI2 ...


,station_id,name,latitude,longitude,elevation,owner,station_type,has_wind,has_water_temperature,has_air_temperature,stdmet_years_found,has_radiation,radiation_years_found
0,45026,"Cook Nuclear Plant Buoy, Stevensville, MI",41.982,-86.619,176.0,Limno Tech,buoy,True,True,True,"2025,2024,2023,2022",False,
1,45168,"South Haven Buoy, MI",42.397,-86.331,177.0,Limno Tech,buoy,True,True,True,"2025,2024,2023,2022",False,
2,45170,"Michigan City Buoy, IN",41.755,-86.968,177.0,Illinois-Indiana Sea Grant and Purdue Civil En...,buoy,True,True,True,"2025,2024,2023,2022",True,"2025,2024,2023,2022"
3,45174,"Wilmette Buoy, IL",42.135,-87.655,176.0,Illinois-Indiana Sea Grant,buoy,True,True,True,"2025,2024,2023,2022",True,"2025,2024,2022"
4,45186,"Waukegan Buoy, IL",42.368,-87.795,176.0,University of Illinois at Urbana-Champaign,buoy,True,True,True,"2025,2024,2023,2022",False,
5,45187,"Winthrop Harbor Buoy, IL",42.491,-87.779,176.0,University of Illinois at Urbana-Champaign,buoy,True,True,True,"2025,2024,2023,2022",False,
6,45198,Chicago Buoy,41.892,-87.563,176.0,Illinois-Indiana Sea Grant and Purdue Civil En...,buoy,True,True,True,"2025,2024,2023,2022",False,
7,BHRI3,"Burns Harbor, IN",41.646,-87.147,180.0,"NWS WFO Chicago, IL (LOT)",fixed,True,False,True,"2025,2024,2023,2022",False,
8,CHII2,"Harrison-Dever Crib, Chicago, IL",41.916,-87.572,176.0,GLERL,fixed,True,False,True,"2025,2024,2023,2022",False,
9,CMTI2,"9087044 - Calumet Harbor, IL",41.730,-87.538,178.5,NOS,fixed,True,False,True,"2025,2024,2023,2022",False,


## Audit the capability table

Review this table before writing the CSV. The `*_years_found` fields show
which annual archives were successfully read and help distinguish an observed
absence from limited archive coverage.


In [6]:
CAPABILITY_COLUMNS = [
    "station_id",
    "name",
    "station_type",
    "has_wind",
    "has_water_temperature",
    "has_air_temperature",
    "has_radiation",
    "stdmet_years_found",
    "radiation_years_found",
]

display(
    buoy_metadata[CAPABILITY_COLUMNS]
)


,station_id,name,station_type,has_wind,has_water_temperature,has_air_temperature,has_radiation,stdmet_years_found,radiation_years_found
0,45026,"Cook Nuclear Plant Buoy, Stevensville, MI",buoy,True,True,True,False,"2025,2024,2023,2022",
1,45168,"South Haven Buoy, MI",buoy,True,True,True,False,"2025,2024,2023,2022",
2,45170,"Michigan City Buoy, IN",buoy,True,True,True,True,"2025,2024,2023,2022","2025,2024,2023,2022"
3,45174,"Wilmette Buoy, IL",buoy,True,True,True,True,"2025,2024,2023,2022","2025,2024,2022"
4,45186,"Waukegan Buoy, IL",buoy,True,True,True,False,"2025,2024,2023,2022",
5,45187,"Winthrop Harbor Buoy, IL",buoy,True,True,True,False,"2025,2024,2023,2022",
6,45198,Chicago Buoy,buoy,True,True,True,False,"2025,2024,2023,2022",
7,BHRI3,"Burns Harbor, IN",fixed,True,False,True,False,"2025,2024,2023,2022",
8,CHII2,"Harrison-Dever Crib, Chicago, IL",fixed,True,False,True,False,"2025,2024,2023,2022",
9,CMTI2,"9087044 - Calumet Harbor, IL",fixed,True,False,True,False,"2025,2024,2023,2022",


### Check a known example

Wilmette station `45174` is a useful audit case because its records include
wind and solar-radiation measurements.


In [7]:
display(
    buoy_metadata.loc[
        buoy_metadata["station_id"] == "45174",
        CAPABILITY_COLUMNS,
    ]
)


,station_id,name,station_type,has_wind,has_water_temperature,has_air_temperature,has_radiation,stdmet_years_found,radiation_years_found
3,45174,"Wilmette Buoy, IL",buoy,True,True,True,True,"2025,2024,2023,2022","2025,2024,2022"


### Optional radiation-file diagnostic

Run this cell when a radiation flag is surprising. It exposes the actual
column names returned by the NDBC radiation reader.


In [8]:
test_radiation = buoys.read_historical_radiation(
    "45174",
    2024,
)

print(test_radiation.columns.tolist())
display(test_radiation.head())


['#YY', 'MM', 'DD', 'hh', 'mm', 'SRAD1', 'SWRAD', 'LWRAD']


,#YY,MM,DD,hh,mm,SRAD1,SWRAD,LWRAD
datetime,,,,,,,,
2024-05-09 17:40:00+00:00,2024,5,9,17,40,77.6,NaN,NaN
2024-05-09 18:50:00+00:00,2024,5,9,18,50,173.4,NaN,NaN
2024-05-09 20:00:00+00:00,2024,5,9,20,0,110.9,NaN,NaN
2024-05-09 21:10:00+00:00,2024,5,9,21,10,40.3,NaN,NaN
2024-05-09 22:20:00+00:00,2024,5,9,22,20,42.2,NaN,NaN


## Export the reusable metadata CSV

The output directory is created if needed. Export is deliberately separated
from the audit cells so the table can be inspected before it overwrites the
existing metadata file.


In [9]:
OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

buoy_metadata.to_csv(
    OUTPUT_FILE,
    index=False,
)

print(
    f"Wrote {len(buoy_metadata)} stations to "
    f"{OUTPUT_FILE}"
)


Wrote 17 stations to data/se_lake_michigan_buoy_metadata.csv
